In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [4]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'Leftover chicken and rice is a fantastic base for many quick, delicious meals. Since both are already cooked, you are already halfway to dinner!\n\nHere are three great ways to transform them:\n\n### 1. Chicken Fried Rice (The Quickest Option)\nThis is arguably the best way to use cold, leftover rice.\n*   **The Vibe:** Classic, savory, and customizable.\n*   **How to do it:** Sauté some garlic, ginger, and any veggies you have (frozen peas/carrots or diced onions work great) in a pan. Push them to the side, crack an egg into the center, and scramble it. Add your cold rice and chicken, then toss with soy sauce, a drizzle of sesame oil, and some green onions. Stir-fry on high heat until crispy and hot.\n\n### 2. Chicken and Rice Casserole (The Comfort Option)\nIf you want something creamy and warming, a casserole is the way to go.\n*   **The Vibe:** Cozy, cheesy, and hearty.\n*   **How to do it:** Mix the chicken and rice in a baking dish with a binder—like a 

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='08e96f67-0d37-43e3-87f8-b1e6b9e063c7'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "easy recipes with leftover chicken and rice"}'}, '__gemini_function_call_thought_signatures__': {'call_727341': 'EnEKbwFpFH0Td8CXfGkJ6OIZlZfnDvE0FFbKD3511bmjWRldU0lxFdujhMTFPn4V3m28fQ6pMK126JZ2YyIBvJ/gDqqFUo4juoHVfDFfAF6xVo9x0sSVxZkvIk02lv1W4Lr8WWyRM16WWDuBROchAU3uSA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0bc7b-c2c8-7c92-add7-85dfed09310f-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'easy recipes with leftover chicken and rice'}, 'id': 'call_727341', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 22, 'tota